In [1]:
# 导入所需要的包
import collections
import math
import os
import shutil
import pandas as pd
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

In [2]:
# 设置数据集路径
data_dir = '../data/cifar-10/'

In [3]:
# 读取csv文件

# 读取fname来给标签字典返回一个文件名
# fname是函数的传入参数，代表传入的csv文件地址
def read_csv_labels(fname):

    with open(fname, 'r') as f:
    # 使用with语句打开文件，确保文件在操作完毕后自动关闭
    # 'r'表示以只读模式打开文件
    # f 是文件对象，用于后续读取文件内容
        
        # 跳过文件第一行
        lines = f.readlines()[1:]
        # f.readlines读取文件所有行，返回一个列表。每行是一个字符串
        # [1:]切片操作，跳过文件第一行，只保留从第二行开始的内容

    # 处理每行数据并分割
    tokens = [l.rstrip().split(',') for l in lines]
    # l.rstrip()去除每行末尾的空格（包括换行符）
    # l.split(';')将每一行按“；”分割成多个字段
    # [... for l in lines] 表示对lines中每一行l执行相应的操作，并返回一个新的列表tokens

    # 创建并返回字典
    return dict(((name, label) for name, label in tokens))
    # dict() 将列表或其他可迭代对象转换为字典
    # ((name, label) for name, label in tokens)：通过生成器表达式遍历 tokens 列表，
    #           将每个 tokens 中的元素拆分成 name 和 label，并将它们作为字典的键值对。
    # 例如，tokens 列表中的每个元素是一个列表 [name, label]，该表达式将其转换为字典中的条目 name: label。

# 函数调用与输出
labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
# os.path.join(data_dir, 'trainLabels.csv')：使用 os.path.join 拼接文件路径，保证跨平台兼容性
# （例如，Windows 和 Linux 路径分隔符不同）。

# 打印结果
print('# 训练样本：', len(labels))
print('# 类别：', len(set(labels.values())))
# len(labels)获取字典labels中键值对的数量，即训练样本数。
# labels.values():获取字典中所有值的列表（即标签）
# set(labels.values()):将标签列表转换为集合，自动去除重复项，因此计算不同标签的数量

# 训练样本： 50000
# 类别： 10


In [4]:
# 提取验证集

# 将文件复制到目录
def copyfile(filename, target_dir):

    # 确保目录存在，没有就创建
    os.makedirs(target_dir, exist_ok = True)

    # 将指定文件filename复制到目标目录
    shutil.copy(filename, target_dir)

# 将验证集从原始的训练集中拆分出来
def reorg_train_valid(data_dir, labels, valid_ratio):
    # valid_ratio验证集所占数据集的比例

    # 得到训练数据集中样本最少的类别中样本数n
    n = collections.Counter(labels.values()).most_common()[-1][1]
    # collections.Counter(labels.values())统计每一个标签类别的样本数量
    #     labels.values() 返回的是一个包含所有标签的列表
    #    （例如，['cat', 'dog', 'cat', 'cat', 'dog']）。Counter 用于统计每个标签的出现频率。
    #     结果可能是Counter({'cat': 3, 'dog': 2})
    # most_common():这是counter对象的方法，用于返回一个按频率排序的元组列表。
    # 每个元组第一个元素是标签，第二个元素是标签的出现次数。调用 most_common() 返回的结果是按频率从高到低排序的：
    # [('cat', 3), ('dog', 2)]
    # [-1] 表示取出 most_common() 返回列表中的最后一个元素
    # [1] 表示获取第二个元素，即出现频率，赋值给n

    # 计算验证集中每个类别的样本数
    n_valid_per_label = max(1, math.floor(n * valid_ratio))
    # 通过 math.floor(n * valid_ratio) 来取出每个类别应该分配到验证集的样本数量，max(1, ...) 确保至少有 1 个验证样本。

    # 初始化标签计数字典
    label_count = {}
    # label_count 是一个字典，用于存储每个类别已经分配到验证集的样本数量。
    # 例如，label_count = {'cat': 1, 'dog': 0} 表示
    #      'cat' 类别已经有 1 个样本被分配到验证集，'dog' 类别没有样本被分配到验证集。

    # 遍历训练集中的文件，按照标签分配到训练集或验证集
    for train_file in os.listdir(os.path.join(data_dir,'train')):
    # os.listdir(os.path.join(data_dir,'train')) 遍历训练集目录中所有样本
        label = labels[train_file.split('.')[0]]
        # train_file.split('.')[0]：
        # 假设文件名格式为 id.png，split('.')[0] 会提取出文件名的id部分（例如：1.png 中的 1）。
        # labels[train_file.split('.')[0]]通过文件名的id部分从labels字典中获取真实的标签
        
        fname = os.path.join(data_dir, 'train', train_file)
        # 获取训练集文件的完整路径
        
        copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'train_valid', label))
        # 将文件复制到新的训练集路径 train_valid_test/train/label 中。文件按照标签被复制到对应的文件夹中。

        if label not in label_count or label_count[label] < n_valid_per_label:
        # 检查当前标签是否已经分配到足够的验证集样本，不够就分配到验证集
            copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'valid', label))
            # 将文件复制到验证集文件夹中
            label_count[label] = label_count.get(label, 0) + 1
            # 更新该标签验证集样本数量
        else:
        # 如果该类别已经有足够的验证集样本，就将文件分配给训练集
            copyfile(fname, os.path.join(data_dir, 'train_valid_test', 'train', label))
            # 将文件复制到训练集文件夹中
    return n_valid_per_label
    # 返回每个标签验证集样本数量

In [5]:
# 在预测期间整理测试集，以方便读取
# 这个函数的目的是在测试期间整理测试集文件，并将它们复制到指定的目标路径。
def reorg_test(data_dir):
    for test_file in os.listdir(os.path.join(data_dir, 'test')):
        copyfile(os.path.join(data_dir, 'test', test_file), 
                 os.path.join(data_dir, 'train_valid_test', 'test', 'unknown'))
        # 复制到目录是一个"未知类别"文件夹

In [6]:
# 最后定义一个函数调用之前的readread_csv_labels,reoreorg_train_valid和reorreorg_test
def reorg_cifar10_data(data_dir, valid_ratio):
    labels = read_csv_labels(os.path.join(data_dir, 'trainLabels.csv'))
    reorg_train_valid(data_dir, labels, valid_ratio)
    reorg_test(data_dir)

In [7]:
# 设置参数并调用刚刚定义的数据读取函数

batch_size = 128
valid_ratio = 0.1
reorg_cifar10_data(data_dir, valid_ratio)